## Concept focus — Decorator stacking order

Stacking order bugs are dangerous because code can look correct while behavior changes silently. Registration, authentication, caching, and logging can all break if the wrappers are applied in the wrong sequence.

```text
@a
@b
@c
def f(): ...

wrap order: a(b(c(f)))
call flow : a -> b -> c -> f
```

### How to think about it
Read decorators in two passes: bottom-up for wrapping and top-down for call flow. That dual reading model prevents a lot of “why is auth missing?” or “why is cache logging weird?” confusion.

### Visual references and further study
- [functools documentation](https://docs.python.org/3/library/functools.html)
- [PEP 318 — decorators](https://peps.python.org/pep-0318/)
- [Real Python — decorators](https://realpython.com/primer-on-python-decorators/)
- [Python Tutor visualizer](https://pythontutor.com/visualize.html)

---

# Module 15 — Decorators, Closures, and functools

## Exercise 15.2 — Stacking order puzzles

Ten cases. Predict the output BEFORE running. Case 7 is the one that matters:
it runs without error and silently disables authentication.
Run:  python ex02_order.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. `functools.wraps` is not optional

In [ ]:
@log_calls
def add(a, b):
    """Add two numbers."""

add.__name__      # 'wrapper'    <- wrong
add.__doc__       # None         <- gone
inspect.signature(add)   # (*args, **kwargs)   <- useless

The wrapper replaced the function, so all of its metadata is the wrapper's.
What breaks, concretely:

- `help()` and every documentation generator
- debuggers and profilers reporting "wrapper" for every decorated function
- **pytest fixture resolution**, which inspects parameter names
- **FastAPI and Pydantic**, which build schemas from signatures
- `singledispatch`, which reads annotations
- any logging that uses `__name__`

In [ ]:
import functools

def log_calls(fn):
    @functools.wraps(fn)          # copies __name__, __doc__, __module__,
    def wrapper(*args, **kwargs): # __qualname__, __dict__, and sets __wrapped__
        return fn(*args, **kwargs)
    return wrapper

`__wrapped__` is what lets `inspect.signature` see through the wrapper to the
real signature. **Always use `wraps`.** There is no case where omitting it is
correct.

---

## Concept 3. Decorators with arguments: three levels

In [ ]:
def retry(attempts=3, delay=1.0):        # 1. the FACTORY takes the arguments
    def decorator(fn):                   # 2. the DECORATOR takes the function
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):    # 3. the WRAPPER takes the call
            for attempt in range(attempts):
                try:
                    return fn(*args, **kwargs)
                except Exception:
                    if attempt == attempts - 1:
                        raise
                    time.sleep(delay)
        return wrapper
    return decorator

@retry(attempts=5)          # note: CALLED. retry(5) returns `decorator`.
def flaky(): ...

`@retry` without parentheses passes the *function* as `attempts`, and the error
appears far away and makes no sense. To support both forms:

```text
def retry(fn=None, *, attempts=3):
    if fn is None:                       # called with arguments
        return functools.partial(retry, attempts=attempts)
    @functools.wraps(fn)
    def wrapper(*a, **kw): ...
    return wrapper

@retry              # works
@retry(attempts=5)  # also works
```


---

## Concept 4. Stacking order

In [ ]:
@a
@b
@c
def f(): ...

# f = a(b(c(f)))

**Bottom-up at definition; top-down at call time.** The decorator closest to the
`def` wraps first, so it is *innermost*, so its wrapper code runs *last* on the
way in.

This ordering matters and gets people:

In [ ]:
@app.route("/admin")        # registers whatever is beneath it
@require_auth               # so the route registered is the AUTHENTICATED one
def admin(): ...

@require_auth               # WRONG ORDER
@app.route("/admin")        # registers the RAW function; auth is never applied
def admin(): ...

The second version registers the undecorated function with the framework and
then wraps a name nobody calls. It looks right, runs fine, and has no
authentication.

Rules of thumb: `@staticmethod`/`@classmethod` outermost; caching outside
logging (so cached calls are not logged as work); registration outermost so it
registers the fully decorated function.

---

## Concept 5. `functools`

### `lru_cache` / `cache`

In [ ]:
@functools.lru_cache(maxsize=128)
def expensive(n: int) -> int: ...

@functools.cache                  # 3.9+: unbounded lru_cache
def fib(n: int) -> int:
    return n if n < 2 else fib(n - 1) + fib(n - 2)

expensive.cache_info()            # hits, misses, maxsize, currsize
expensive.cache_clear()

Five things to know before using it:

1. **Arguments must be hashable.** A list argument raises `TypeError`.
2. **Equal-but-distinct arguments can collide.** `1`, `1.0` and `True` are equal
   and hash equally (Module 03), so a function that treats them differently can
   get the wrong cached answer. The exact behaviour is subtler than it looks —
   `lru_cache` has a fast path for a single `int` or `str` argument, so the real
   grouping is not the one you would predict. Exercise 15.3 measures it. The
   safe rule: if your function's behaviour depends on the *type* of a numeric
   argument, do not cache it by that argument.
3. **`f(1)` and `f(x=1)` are different entries.** Same call, two cache slots.
4. **It keeps a strong reference to every argument and result.** `@cache` on a
   method keeps every instance alive forever — a genuine and common memory leak.
   Use `maxsize`, or `cached_property`, or a `WeakValueDictionary`.
5. **Only cache pure functions.** A cached function with side effects performs
   them once and silently skips them thereafter.

### `partial`

In [ ]:
from functools import partial
int2 = partial(int, base=2)
int2("1010")                       # 10
sorted(rows, key=partial(get_field, "name"))

`partial` beats a lambda for a callback: it has a useful `repr`, it is
picklable (so it works with `multiprocessing`, Module 21), and it does not
capture variables by reference — which sidesteps Module 04's late-binding trap.

### `singledispatch`

In [ ]:
@functools.singledispatch
def serialise(obj) -> str:
    raise TypeError(f"cannot serialise {type(obj).__name__}")

@serialise.register
def _(obj: datetime) -> str: return obj.isoformat()

@serialise.register
def _(obj: Decimal) -> str: return str(obj)

Type-based dispatch without an `isinstance` chain, and — importantly —
**open for extension**: a third party can register a handler for their own type
without touching your code. This is the Visitor pattern, dissolved (Module 12).

`singledispatchmethod` does the same for methods.

### `cached_property`, `total_ordering`, `reduce`

In [ ]:
@functools.cached_property        # Module 08: computed once, stored in __dict__
def stats(self): ...

@functools.total_ordering         # Module 09: fills in <=, >, >= from < and ==
class Version: ...

functools.reduce(operator.mul, nums, 1)     # rarely clearer than a loop

`reduce` is worth knowing and rarely worth using. `sum`, `math.prod`,
`itertools.accumulate` and an explicit loop are all clearer.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `@` is sugar
- Section 2: `functools.wraps` is not optional
- Section 3: Decorators with arguments: three levels
- Section 4: Stacking order
- Section 5: `functools`
- Section 6: `contextlib`
- Section 7: Class-based decorators, and when to use one

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import functools
from typing import Any, Callable

TRACE: list[str] = []

---

## `make`

A decorator that records when it WRAPS and when it CALLS.

In [ ]:
def make(label: str) -> Callable[[Callable], Callable]:  # type: ignore[type-arg]
    """A decorator that records when it WRAPS and when it CALLS."""
    def decorator(fn: Callable) -> Callable:  # type: ignore[type-arg]
        TRACE.append(f"wrap:{label}")
        @functools.wraps(fn)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            TRACE.append(f"enter:{label}")
            result = fn(*args, **kwargs)
            TRACE.append(f"exit:{label}")
            return result
        return wrapper
    return decorator

---

## `q01`

_q01_

In [ ]:
def q01() -> None:
    # PREDICTION: what is in TRACE after DEFINITION, before any call?
    TRACE.clear()

    @make("a")
    @make("b")
    @make("c")
    def f() -> str:
        TRACE.append("body")
        return "done"

    print("q01 after definition:", TRACE.copy())
    TRACE.clear()
    f()
    print("q01 after call:      ", TRACE.copy())

---

## `q02`

_q02_

In [ ]:
def q02() -> None:
    # PREDICTION: how many times does "wrap:x" appear for three calls?
    TRACE.clear()

    @make("x")
    def g() -> None: ...

    g(); g(); g()
    print("q02:", [t for t in TRACE if t.startswith("wrap")])

---

## `q03`

_q03_

In [ ]:
def q03() -> None:
    # PREDICTION: does this even work? What is __name__?
    def no_wraps(fn):  # type: ignore[no-untyped-def]
        def wrapper(*a, **kw):  # type: ignore[no-untyped-def]
            return fn(*a, **kw)
        return wrapper

    @no_wraps
    def documented() -> None:
        """A docstring."""

    print("q03:", documented.__name__, repr(documented.__doc__))

---

## `q04`

_q04_

In [ ]:
def q04() -> None:
    # PREDICTION: what does inspect.signature report for each?
    import inspect

    def bare(fn):  # type: ignore[no-untyped-def]
        def wrapper(*a, **kw):  # type: ignore[no-untyped-def]
            return fn(*a, **kw)
        return wrapper

    def wrapped(fn):  # type: ignore[no-untyped-def]
        @functools.wraps(fn)
        def wrapper(*a, **kw):  # type: ignore[no-untyped-def]
            return fn(*a, **kw)
        return wrapper

    @bare
    def one(a: int, b: str = "x") -> None: ...

    @wrapped
    def two(a: int, b: str = "x") -> None: ...

    print("q04 bare:   ", inspect.signature(one))
    print("q04 wrapped:", inspect.signature(two))

---

## `q05`

_q05_

In [ ]:
def q05() -> None:
    # PREDICTION: what error, and why does the message not mention parentheses?
    def retry(attempts: int = 3):  # type: ignore[no-untyped-def]
        def decorator(fn):  # type: ignore[no-untyped-def]
            @functools.wraps(fn)
            def wrapper(*a, **kw):  # type: ignore[no-untyped-def]
                return fn(*a, **kw)
            return wrapper
        return decorator

    try:
        @retry                       # note: NO parentheses
        def flaky() -> str:
            return "ok"
        print("q05:", flaky())
    except Exception as exc:
        print("q05:", type(exc).__name__, exc)

---

## `q06`

_q06_

In [ ]:
def q06() -> None:
    # PREDICTION: does the cache see the log, or does the log see the cache?
    calls = {"real": 0, "logged": 0}

    def logging_deco(fn):  # type: ignore[no-untyped-def]
        @functools.wraps(fn)
        def wrapper(*a, **kw):  # type: ignore[no-untyped-def]
            calls["logged"] += 1
            return fn(*a, **kw)
        return wrapper

    @functools.cache
    @logging_deco
    def cached_outside(n: int) -> int:
        calls["real"] += 1
        return n * 2

    for _ in range(3):
        cached_outside(5)
    print("q06 cache outside log:", dict(calls))

    calls["real"] = calls["logged"] = 0

    @logging_deco
    @functools.cache
    def cache_inside(n: int) -> int:
        calls["real"] += 1
        return n * 2

    for _ in range(3):
        cache_inside(5)
    print("q06 log outside cache:", dict(calls))

---

## `q07`

_q07_

In [ ]:
def q07() -> None:
    # THE IMPORTANT ONE. Predict what each route registry ends up holding.
    routes: dict[str, Callable] = {}  # type: ignore[type-arg]

    def route(path: str):  # type: ignore[no-untyped-def]
        def register(fn):  # type: ignore[no-untyped-def]
            routes[path] = fn
            return fn
        return register

    def require_auth(fn):  # type: ignore[no-untyped-def]
        @functools.wraps(fn)
        def wrapper(user: str | None = None, *a, **kw):  # type: ignore[no-untyped-def]
            if user is None:
                raise PermissionError("not authenticated")
            return fn(user, *a, **kw)
        return wrapper

    @route("/right")
    @require_auth
    def right(user: str) -> str:
        return f"admin page for {user}"

    @require_auth
    @route("/wrong")
    def wrong(user: str) -> str:
        return f"admin page for {user}"

    for path in ("/right", "/wrong"):
        try:
            result = routes[path](None)          # an UNAUTHENTICATED request
            print(f"q07 {path}: SERVED -> {result!r}")
        except PermissionError as exc:
            print(f"q07 {path}: blocked ({exc})")

---

## `q08`

_q08_

In [ ]:
def q08() -> None:
    # PREDICTION: does the decorator see the method or the staticmethod object?
    def show_type(fn):  # type: ignore[no-untyped-def]
        print(f"q08 decorator received: {type(fn).__name__}")
        return fn

    class C:
        @show_type
        @staticmethod
        def inner_static() -> None: ...

        @staticmethod
        @show_type
        def outer_static() -> None: ...

---

## `q09`

_q09_

In [ ]:
def q09() -> None:
    # PREDICTION: what does each print?
    def add_attr(name: str, value: Any):  # type: ignore[no-untyped-def]
        def decorator(fn):  # type: ignore[no-untyped-def]
            @functools.wraps(fn)
            def wrapper(*a, **kw):  # type: ignore[no-untyped-def]
                return fn(*a, **kw)
            setattr(wrapper, name, value)
            return wrapper
        return decorator

    @add_attr("outer", 1)
    @add_attr("inner", 2)
    def f() -> None: ...

    print("q09:", getattr(f, "outer", "MISSING"), getattr(f, "inner", "MISSING"))

---

## `q10`

_q10_

In [ ]:
def q10() -> None:
    # PREDICTION: how many entries does the cache have after these four calls?
    @functools.cache
    def f(n: Any) -> Any:
        return n

    f(1); f(1.0); f(True); f(n=1)
    print("q10 cache entries:", f.cache_info().currsize)

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    for fn in [q01, q02, q03, q04, q05, q06, q07, q08, q09, q10]:
        fn()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.